# Local-loopback development

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import the exact published
verifier **2.0.0**.

Verification returns facts; the host still owns trusted inputs, authorization,
replay and revocation. Only safe semantic results are displayed; generated keys,
signatures and nonces stay in memory.

## Load the published verifier

Check the package identity and the portable backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import {
  assembleCompact,
  assembleLocalLoopbackHttpCompact,
  BACKEND_ID,
  checkEnvelope,
  checkLocalLoopbackHttpEnvelope,
  type ExpectedRequest,
  grantSigningInput,
  jwkFromPublicKey,
  localLoopbackHttpProofSigningInput,
  localLoopbackHttpUriNormalize,
  type SigningInput,
  strUtf8,
  type Tagged,
  thumbprint,
  utf8Str,
} from "npm:@bounded-authority-protocol/verifier@2.0.0";
import {
  generateKeyPairSync,
  type KeyObject,
  randomUUID,
  sign,
} from "node:crypto";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with {
  type: "json",
};
import assert from "node:assert/strict";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(
  JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }),
);

{"version":"2.0.0","backend":"portable"}


## Prepare demonstration keys

Generate fresh issuer and holder keys. The helpers expose public bytes and
assemble the standard JWS signing input.

In [ ]:
const publicKeyBytes = (key: KeyObject): Uint8Array =>
  Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const signingBytes = (input: SigningInput): Uint8Array =>
  Buffer.concat([
    input.protectedSegment,
    Buffer.from("."),
    input.payloadSegment,
  ]);
const issuer = generateKeyPairSync("ed25519");
const holder = generateKeyPairSync("ed25519");
assert.equal(publicKeyBytes(issuer.publicKey).length, 32);
assert.equal(publicKeyBytes(holder.publicKey).length, 32);
console.log(
  JSON.stringify({
    issuerKeyBytes: publicKeyBytes(issuer.publicKey).length,
    holderKeyBytes: publicKeyBytes(holder.publicKey).length,
  }),
);

{"issuerKeyBytes":32,"holderKeyBytes":32}


## Issue one grant

Bind the grant to the holder and the readings.upload operation.


In [ ]:
const grantInput = grantSigningInput({
  keyId: "k1",
  issuer: "https://issuer.example",
  grantId: "grant-n",
  audiences: ["https://telemetry.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 9_000,
  holderThumbprint: thumbprint(
    jwkFromPublicKey(publicKeyBytes(holder.publicKey)),
  ),
  operations: [{ name: "readings.upload", selectors: ["all"] }],
});
assert.ok(grantInput.ok);
const grant = assembleCompact(
  grantInput.value,
  sign(null, signingBytes(grantInput.value), issuer.privateKey),
);
assert.ok(grant.ok);

console.log(JSON.stringify({ ok: grant.ok }));

{"ok":true}


## Sign the request

The holder binds the exact method, target, arguments, invocation and server
challenge.

In [ ]:
const args: Tagged = { t: "object", v: new Map() };
const invocationId = randomUUID();
const proofInput = localLoopbackHttpProofSigningInput({
  holderPublicKey: publicKeyBytes(holder.publicKey),
  proofId: "proof-n",
  method: "PUT",
  targetUri: "http://127.0.0.1:3210/readings",
  issuedAt: 1_100,
  invocationId,
  operation: "readings.upload",
  grantCompact: grant.value,
  castArguments: args,
  nonce: "srv-nonce-5c1e", // the value your server issued
});
assert.ok(proofInput.ok);
const proof = assembleLocalLoopbackHttpCompact(
  proofInput.value,
  sign(null, signingBytes(proofInput.value), holder.privateKey),
);
assert.ok(proof.ok);

console.log(JSON.stringify({ ok: proof.ok }));

{"ok":true}


## Verify the expected request

Supply the already-trusted issuer key and the context the host expects.


In [ ]:
const expected: ExpectedRequest = {
  trustedIssuer: { keyId: "k1", publicKey: publicKeyBytes(issuer.publicKey) },
  issuer: "https://issuer.example",
  audience: "https://telemetry.example",
  method: "PUT",
  targetUri: "http://127.0.0.1:3210/readings",
  invocationId,
  operation: "readings.upload",
  castArguments: args,
  evaluationTime: 1_120,
  clockSkew: 30,
  proofMaxAge: 120,
  nonce: { kind: "required", value: "srv-nonce-5c1e" },
};
const verified = checkLocalLoopbackHttpEnvelope(
  grant.value,
  proof.value,
  expected,
);
assert.ok(verified.ok);
assert.equal(verified.value.operation, "readings.upload");
console.log(
  JSON.stringify({
    ok: verified.ok,
    operation: verified.value.operation,
    authorization: verified.value.authorization,
  }),
);

{"ok":true,"operation":"readings.upload","authorization":"not_evaluated"}


## Refuse a different challenge

Keep the signed request unchanged and require a different challenge.


In [ ]:
const wrongNonce = checkLocalLoopbackHttpEnvelope(grant.value, proof.value, {
  ...expected,
  nonce: { kind: "required", value: "other" },
});
assert.deepEqual(wrongNonce, { ok: false });
console.log(JSON.stringify(wrongNonce));

{"ok":false}


## Refuse the standard profile

Loopback proofs are explicit and cannot pass the standard HTTPS profile.


In [ ]:
const standardProfile = checkEnvelope(grant.value, proof.value, expected);
assert.deepEqual(standardProfile, { ok: false });
console.log(JSON.stringify(standardProfile));


{"ok":false}


## Require a challenge policy

The loopback profile refuses a host context that does not require the challenge.


In [ ]:
const missingPolicy = checkLocalLoopbackHttpEnvelope(grant.value, proof.value, {
  ...expected,
  nonce: { kind: "not_required" },
});
assert.deepEqual(missingPolicy, { ok: false });
console.log(JSON.stringify(missingPolicy));

{"ok":false}


## Normalize literal loopback addresses

The profile accepts literal loopback addresses; hostnames and remote addresses
refuse.

In [ ]:
const normalized = localLoopbackHttpUriNormalize(
  strUtf8("http://[::1]:80/a/../readings"),
);
assert.ok(normalized.ok);
assert.equal(utf8Str(normalized.value), "http://[::1]/readings");
assert.deepEqual(
  localLoopbackHttpUriNormalize(strUtf8("http://localhost/readings")),
  { ok: false },
);
assert.deepEqual(
  localLoopbackHttpUriNormalize(strUtf8("http://192.0.2.1/readings")),
  { ok: false },
);

console.log(JSON.stringify({
  normalized: utf8Str(normalized.value),
  hostname:
    localLoopbackHttpUriNormalize(strUtf8("http://localhost/readings")).ok,
  remote:
    localLoopbackHttpUriNormalize(strUtf8("http://192.0.2.1/readings")).ok,
}));

{"normalized":"http://[::1]/readings","hostname":false,"remote":false}


## Finish

No private material was written to disk. Shut down the kernel to discard the
demonstration keys. Continue with the
[getting started guide](../guides/getting-started.md).